In [8]:
import csv
import json
import math
import re
import statistics
from pathlib import Path

In [9]:



# ============================================================
# SETTINGS
# ============================================================

folder = Path.cwd()
data_folder = folder / "csv_data"
output_file = folder / "index.html"

RMS_LIMIT_PN = 25.0
RMS_COLUMN = "elasticity_rms_bruker_pN"
SLOPE_COLUMN = "linear_slope_bruker_N_per_m"
E_COLUMN = "youngs_modulus_MPa"

CONDITION_ORDER = [
    "Live",
    "20 mg/L",
    "320 mg/L",
    "Dead",
]

FOLDER_PATTERN = re.compile(
    r"^(?P<condition>Live|20|320|Dead)_cell(?P<cell>\d+)_crop(?P<crop>\d+)$",
    re.IGNORECASE,
)

CONDITION_LABELS = {
    "live": "Live",
    "20": "20 mg/L",
    "320": "320 mg/L",
    "dead": "Dead",
}


# ============================================================
# HELPERS
# ============================================================

def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def get_points(rows, column, used_column=None):
    points = []

    for row in rows:
        if used_column and row.get(used_column) != "1":
            continue

        try:
            x = float(row["x_nm"])
            y = float(row[column])
        except (KeyError, TypeError, ValueError):
            continue

        if math.isfinite(x) and math.isfinite(y):
            points.append([x, y])

    if used_column:
        points.sort(key=lambda point: point[0])

    return points


def curve_key(row):
    return (
        row.get("filename", ""),
        row.get("curve_number", ""),
        row.get("position_index", ""),
    )


def safe_float(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None

    return number if math.isfinite(number) else None


def bacterial_spring_constant_nN_per_um(row):
    """kB = abs(JPK Linear Fit slope), in nN/um (1 N/m = 1000 nN/um).

    The original CSV may contain an obsolete compliance-corrected kB.
    Always recalculate from linear_slope_bruker_N_per_m instead.
    """
    slope = safe_float(row.get(SLOPE_COLUMN))
    return abs(slope) * 1000.0 if slope is not None else None


def fit_range_nm(row, minimum_column, maximum_column):
    """Full fitted X interval width in nm."""
    xmin = safe_float(row.get(minimum_column))
    xmax = safe_float(row.get(maximum_column))
    if xmin is None or xmax is None:
        return None
    return abs(xmax - xmin)


def fitted_interval_width_to_contact_nm(row, contact_x_nm):
    """Width of a fitted X interval from its minimum to a chosen contact point.

    The contact point is clipped to the fitted interval so the reported width
    never extends outside the plotted fit domain.
    """
    xmin = safe_float(row.get("elasticity_x_min_used_nm"))
    xmax = safe_float(row.get("elasticity_x_max_used_nm"))
    if xmin is None or xmax is None or contact_x_nm is None:
        return None

    low, high = sorted((xmin, xmax))
    boundary = min(max(contact_x_nm, low), high)
    return max(0.0, boundary - low)


def elasticity_range_to_preprocessing_contact_nm(row):
    """Indentation-side width relative to the processed-curve contact at X = 0 nm."""
    return fitted_interval_width_to_contact_nm(row, 0.0)



def elasticity_range_to_fitted_contact_nm(row):
    """Indentation-side width relative to the fitted elasticity contact point xc."""
    xc = safe_float(row.get("contact_point_x_nm"))
    return fitted_interval_width_to_contact_nm(row, xc)


def curve_values_with_derived_fields(row):
    """Recompute displayed parameters from JPK source fields, not old kB."""
    if not row:
        return {}
    values = dict(row)
    kb = bacterial_spring_constant_nN_per_um(values)
    values["bacterial_spring_constant_nN_per_um"] = kb if kb is not None else ""
    values["linear_fit_range_nm"] = fit_range_nm(
        values, "linear_x_min_used_nm", "linear_x_max_used_nm"
    )
    values["elasticity_fit_range_nm"] = fit_range_nm(
        values, "elasticity_x_min_used_nm", "elasticity_x_max_used_nm"
    )
    values["elasticity_fit_range_to_preprocessing_contact_nm"] = (
        elasticity_range_to_preprocessing_contact_nm(values)
    )
    values["elasticity_fit_range_to_fitted_contact_nm"] = (
        elasticity_range_to_fitted_contact_nm(values)
    )
    # Do not expose the old compliance-corrected value from the CSV.
    values.pop("bacterial_spring_constant_N_per_m", None)
    return values


def median_absolute_deviation(values):
    clean = [
        float(value)
        for value in values
        if value is not None and math.isfinite(float(value))
    ]

    if not clean:
        return None

    median_value = statistics.median(clean)
    deviations = [abs(value - median_value) for value in clean]
    return statistics.median(deviations)


def format_stat(value, digits=4):
    if value is None:
        return ""

    try:
        number = float(value)
    except (TypeError, ValueError):
        return ""

    if not math.isfinite(number):
        return ""

    return f"{number:.{digits}f}"


# ============================================================
# LOAD CURVES FOR THE INTERACTIVE DASHBOARD
# ============================================================

datasets = []

# A summary row is eligible for the aggregate table only when it matches an
# existing, nonempty per-curve fit-points CSV in the SAME crop folder.
# Starting from the fit-points files prevents leftover/rejected summary rows
# from entering the cell and condition statistics.
matched_curve_summaries = []
total_summary_rows = 0

curve_folders = sorted({
    path.parent
    for path in data_folder.rglob("AFM_fit_points_curve_*.csv")
})

for curve_folder in curve_folders:
    summary_file = curve_folder / "AFM_jython_summary.csv"

    if not summary_file.exists():
        raise FileNotFoundError(
            f"Missing summary CSV in {curve_folder}"
        )

    summary_rows = read_csv(summary_file)
    total_summary_rows += len(summary_rows)

    # Preserve duplicate entries for explicit ambiguity detection instead of
    # silently overwriting one when constructing a dictionary.
    summary_by_key = {}
    for row in summary_rows:
        if not row.get("filename") or not row.get("position_index"):
            continue
        summary_by_key.setdefault(curve_key(row), []).append(row)

    counted_keys = set()
    matched_count = 0

    # A few older point exports have an incorrect curve_number, even
    # when their filename and JPK position_index identify the curve.
    # Use a unique filename/position match only as a validated fallback.
    summary_by_identity = {}
    for row in summary_rows:
        identity = (row.get("filename", ""), row.get("position_index", ""))
        summary_by_identity.setdefault(identity, []).append(row)

    curves = []

    for path in sorted(
        curve_folder.glob("AFM_fit_points_curve_*.csv")
    ):
        rows = read_csv(path)

        if not rows:
            print("Skipped empty file:", path.name)
            continue

        first = rows[0]
        # First try the full JPK identity: filename, curve number, position.
        exact_matches = summary_by_key.get(curve_key(first), [])
        matched_summary = exact_matches[0] if len(exact_matches) == 1 else None
        if len(exact_matches) > 1:
            print("Ambiguous duplicate summary rows, not matched:", path.name)
        elif not exact_matches:
            # Some historic fit-point exports misstate curve_number. Only
            # recover by filename + position when the match is unambiguous.
            identity = (first.get("filename", ""), first.get("position_index", ""))
            candidates = summary_by_identity.get(identity, [])
            if len(candidates) == 1:
                matched_summary = candidates[0]
                print(
                    "Curve-number mismatch recovered:", path.name,
                    "| fit-point CSV:", first.get("curve_number"),
                    "| JPK summary:", matched_summary.get("curve_number"),
                )
            elif len(candidates) > 1:
                print("Ambiguous summary rows, not matched:", path.name)

        values = curve_values_with_derived_fields(matched_summary or {})
        if not values:
            print("No matching summary row:", path.name)
        else:
            # Count a JPK curve once even if its per-curve export is
            # inadvertently copied or duplicated within the crop folder.
            key = curve_key(matched_summary)
            if key not in counted_keys:
                counted_keys.add(key)
                matched_curve_summaries.append((curve_folder, matched_summary))
                matched_count += 1
            else:
                print("Duplicate fit-point CSV, excluded from summary:", path.name)

        curves.append({
            "number": (
                matched_summary.get("curve_number", first["curve_number"])
                if matched_summary else first["curve_number"]
            ),
            "position": first["position_index"],
            "filename": first["filename"],
            "measured": get_points(
                rows,
                "measured_y_nN",
            ),
            "linear": get_points(
                rows,
                "linear_fit_y_nN",
                "linear_fit_used",
            ),
            "elasticity": get_points(
                rows,
                "elasticity_fit_y_nN",
                "elasticity_fit_used",
            ),
            "values": values,
        })

    curves.sort(
        key=lambda curve: (
            int(curve["number"]),
            int(curve["position"]),
        )
    )

    if curves:
        datasets.append({
            "name": curve_folder.relative_to(data_folder).as_posix(),
            "curves": curves,
        })
    print(
        "Summary match count:", curve_folder.name,
        "|", matched_count, "matched per-curve CSVs",
        "|", max(0, len(summary_rows) - matched_count),
        "summary entries not used",
    )

if not datasets:
    raise SystemExit(
        "No curves found. Put your CSV files inside csv_data."
    )


# ============================================================
# BUILD THE SUMMARY TABLE DATA
# ============================================================
#
# Crop folders for the same condition and cell are pooled.
# Example:
#   Live_cell1_crop1
#   Live_cell1_crop2
# become:
#   Live, Cell 1
#
# Inclusion criteria for the aggregate summary table:
#   (1) A real, nonempty per-curve fit-points CSV exists in the same crop.
#   (2) The curve has a unique matching JPK summary row.
#   (3) The existing JPK elasticity RMS <= 25 pN table filter remains.
#   Rows present only in AFM_jython_summary.csv are NEVER included.
# kB is always recalculated from the original JPK Linear Fit slope,
# without cantilever-compliance correction.
#
# Per cell:
#   n, median kB, MAD kB, median E, MAD E
#
# Per condition:
#   mean of cell medians and sample SD of cell medians
#
# ============================================================

summary_groups = {}

# IMPORTANT: Build statistics from the accepted fit-point file inventory,
# not from an independent pass over every row in AFM_jython_summary.csv.
for curve_folder, row in matched_curve_summaries:
    match = FOLDER_PATTERN.match(curve_folder.name)
    if match is None:
        continue

    condition = CONDITION_LABELS[match.group("condition").lower()]
    cell_number = int(match.group("cell"))
    group_key = (condition, cell_number)
    summary_groups.setdefault(group_key, [])

    rms = safe_float(row.get(RMS_COLUMN))
    kb = bacterial_spring_constant_nN_per_um(row)
    youngs = safe_float(row.get(E_COLUMN))

    if rms is None or rms > RMS_LIMIT_PN:
        continue

    if kb is None and youngs is None:
        continue

    summary_groups[group_key].append({
        "rms": rms,
        "kb": kb,
        "E": youngs,
        "curve_number": row.get("curve_number", ""),
        "position_index": row.get("position_index", ""),
        "source_folder": curve_folder.name,
    })

cell_stats = {}

for key, rows in summary_groups.items():
    kb_values = [
        row["kb"]
        for row in rows
        if row["kb"] is not None
    ]

    e_values = [
        row["E"]
        for row in rows
        if row["E"] is not None
    ]

    cell_stats[key] = {
        "n": len(rows),
        "kb_median": (
            statistics.median(kb_values)
            if kb_values
            else None
        ),
        "kb_mad": median_absolute_deviation(kb_values),
        "e_median": (
            statistics.median(e_values)
            if e_values
            else None
        ),
        "e_mad": median_absolute_deviation(e_values),
    }


condition_stats = {}

for condition in CONDITION_ORDER:
    cells = sorted(
        cell_number
        for group_condition, cell_number in cell_stats
        if group_condition == condition
    )

    kb_medians = [
        cell_stats[(condition, cell)]["kb_median"]
        for cell in cells
        if cell_stats[(condition, cell)]["kb_median"] is not None
    ]

    e_medians = [
        cell_stats[(condition, cell)]["e_median"]
        for cell in cells
        if cell_stats[(condition, cell)]["e_median"] is not None
    ]

    condition_stats[condition] = {
        "cells": cells,
        "kb_mean_of_medians": (
            statistics.mean(kb_medians)
            if kb_medians
            else None
        ),
        "kb_sd": (
            statistics.stdev(kb_medians)
            if len(kb_medians) >= 2
            else None
        ),
        "e_mean_of_medians": (
            statistics.mean(e_medians)
            if e_medians
            else None
        ),
        "e_sd": (
            statistics.stdev(e_medians)
            if len(e_medians) >= 2
            else None
        ),
    }


summary_rows_html = []

for condition in CONDITION_ORDER:
    group = condition_stats[condition]
    cells = group["cells"]

    if not cells:
        continue

    rowspan = len(cells)

    for index, cell_number in enumerate(cells):
        stats = cell_stats[(condition, cell_number)]

        condition_cell = ""
        kb_group_cells = ""
        e_group_cells = ""

        if index == 0:
            condition_cell = (
                f'<td class="summary-condition" rowspan="{rowspan}">'
                f'{condition}</td>'
            )

            kb_group_cells = (
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["kb_mean_of_medians"], 2)}</td>'
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["kb_sd"], 2)}</td>'
            )

            e_group_cells = (
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["e_mean_of_medians"], 2)}</td>'
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["e_sd"], 2)}</td>'
            )

        summary_rows_html.append(
            "<tr>"
            + condition_cell
            + f"<td>Cell {cell_number}</td>"
            + f'<td>{stats["n"]}</td>'
            + f'<td>{format_stat(stats["kb_median"], 4)}</td>'
            + f'<td>{format_stat(stats["kb_mad"], 4)}</td>'
            + kb_group_cells
            + f'<td>{format_stat(stats["e_median"], 4)}</td>'
            + f'<td>{format_stat(stats["e_mad"], 4)}</td>'
            + e_group_cells
            + "</tr>"
        )


if summary_rows_html:
    summary_table_html = f"""
    <div class="card summary-card">
        <h2>AFM summary, JPK elasticity RMS ≤ {RMS_LIMIT_PN:g} pN</h2>

        <div class="summary-table-wrapper">
            <table class="summary-table">
                <thead>
                    <tr>
                        <th rowspan="2">Condition</th>
                        <th rowspan="2">Cell</th>
                        <th>RMS ≤ {RMS_LIMIT_PN:g} pN</th>
                        <th colspan="4">kB (nN/µm)</th>
                        <th colspan="4">E (MPa)</th>
                    </tr>
                    <tr>
                        <th>n</th>
                        <th>Median</th>
                        <th>MAD</th>
                        <th>Mean of medians</th>
                        <th>SD</th>
                        <th>Median</th>
                        <th>MAD</th>
                        <th>Mean of medians</th>
                        <th>SD</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(summary_rows_html)}
                </tbody>
            </table>
        </div>
    </div>
    """
else:
    summary_table_html = f"""
    <div class="card summary-card">
        <h2>AFM summary, JPK elasticity RMS ≤ {RMS_LIMIT_PN:g} pN</h2>
        <p>No matching condition/cell crop folders were found.</p>
    </div>
    """


# ============================================================
# HTML PAGE
# ============================================================

html = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>AFM CSV Dashboard</title>

    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: #f4f5f7;
            color: #202124;
        }

        header {
            padding: 20px 28px;
            background: white;
            border-bottom: 1px solid #ddd;
        }

        h1 {
            margin: 0;
            font-size: 24px;
        }

        header p {
            margin-bottom: 0;
            color: #666;
        }

        .controls,
        .card {
            background: white;
            padding: 20px;
            border-radius: 8px;
        }

        .controls {
            margin: 20px;
            display: flex;
            align-items: center;
            flex-wrap: wrap;
            gap: 12px;
        }

        select,
        button {
            padding: 9px 12px;
            font-size: 14px;
        }

        select {
            min-width: 200px;
            max-width: 100%;
        }

        button {
            cursor: pointer;
        }

        .layout {
            display: grid;
            grid-template-columns: minmax(0, 1fr) 360px;
            gap: 20px;
            margin: 20px;
        }

        .card {
            min-width: 0;
        }

        #plot {
            height: 600px;
        }

        h2 {
            margin-top: 0;
            font-size: 20px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }

        #values td {
            padding: 10px 4px;
            border-bottom: 1px solid #eee;
        }

        #values td:nth-child(2) {
            text-align: right;
            font-weight: bold;
        }

        #values td:last-child {
            color: #666;
            padding-left: 10px;
            white-space: nowrap;
        }

        #filename,
        #status {
            font-size: 13px;
            color: #666;
            line-height: 1.5;
            overflow-wrap: anywhere;
        }

        .summary-card {
            margin: 20px;
        }

        .summary-table-wrapper {
            overflow-x: auto;
        }

        .summary-table {
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }

        .summary-table th,
        .summary-table td {
            padding: 9px 10px;
            border: 1px solid #d9dde2;
            text-align: center;
            vertical-align: middle;
            white-space: nowrap;
        }

        .summary-table th {
            background: #f5f6f7;
            font-weight: 700;
        }

        .summary-table .summary-condition {
            font-weight: 700;
            background: #fafafa;
        }

        .summary-table .summary-group-value {
            font-weight: 600;
        }

        @media (max-width: 900px) {
            .layout {
                grid-template-columns: 1fr;
            }

            #plot {
                height: 450px;
            }
        }
    </style>
</head>

<body>
    <header>
        <h1>AFM CSV Dashboard</h1>
        <p>Force curves and saved fit results</p>
    </header>

    <div class="controls">
        <label for="folder">Folder</label>
        <select id="folder"></select>

        <label for="curve">Curve</label>
        <select id="curve"></select>

        <button id="previous">Previous</button>
        <button id="next">Next</button>
    </div>

    <div class="layout">
        <div class="card">
            <div id="plot"></div>
        </div>

        <div class="card">
            <h2>Curve values</h2>

            <table>
                <tbody id="values"></tbody>
            </table>

            <p id="status"></p>
            <p id="filename"></p>
        </div>
    </div>

    __SUMMARY_TABLE__

    <script>
        const datasets = __DATA__;

        const folderSelect = document.getElementById("folder");
        const curveSelect = document.getElementById("curve");
        const valuesTable = document.getElementById("values");
        const status = document.getElementById("status");
        const filenameDisplay = document.getElementById("filename");

        const fields = [
            ["Linear slope", "linear_slope_bruker_N_per_m", "N/m"],
            ["Bacterial spring constant", "bacterial_spring_constant_nN_per_um", "nN/µm"],
            ["Linear RMSD custom", "linear_rmsd_custom_pN", "pN"],
            ["Linear X minimum", "linear_x_min_used_nm", "nm"],
            ["Linear X maximum", "linear_x_max_used_nm", "nm"],
            ["Linear fit range (nm)", "linear_fit_range_nm", ""],
            ["Young's modulus", "youngs_modulus_MPa", "MPa"],
            ["JPK Elasticity RMS", "elasticity_rms_bruker_pN", "pN"],
            ["Elasticity RMSD custom", "elasticity_rmsd_custom_pN", "pN"],
            ["Elasticity X minimum", "elasticity_x_min_used_nm", "nm"],
            ["Elasticity X maximum", "elasticity_x_max_used_nm", "nm"],
            ["Elasticity fit range (nm)", "elasticity_fit_range_nm", ""],
            ["Elasticity fit range to preprocessing contact (nm)", "elasticity_fit_range_to_preprocessing_contact_nm", ""],
            ["Elasticity fit range to fitted contact (nm)", "elasticity_fit_range_to_fitted_contact_nm", ""],
        ];

        function formatNumber(value) {
            if (value == null || String(value).trim() === "") {
                return "N/A";
            }

            const number = Number(value);

            return Number.isFinite(number)
                ? String(Number(number.toPrecision(6)))
                : "N/A";
        }

        function percentageDifference(value1, value2) {
            const a = Number(value1);
            const b = Number(value2);

            if (!Number.isFinite(a) || !Number.isFinite(b)) {
                return "N/A";
            }

            const mean = (Math.abs(a) + Math.abs(b)) / 2;

            if (mean === 0) {
                return "0";
            }

            const difference = Math.abs(a - b) / mean * 100;

            return String(Number(difference.toPrecision(6)));
        }

        function appendValueRow(label, value, unit) {
            const row = document.createElement("tr");

            for (const text of [label, value, unit]) {
                const cell = document.createElement("td");
                cell.textContent = text;
                row.appendChild(cell);
            }

            valuesTable.appendChild(row);
        }

        function makeTrace(points, name, color, mode) {
            return {
                x: points.map(point => point[0]),
                y: points.map(point => point[1]),
                type: "scatter",
                mode: mode,
                name: name,
                marker: { color: color, size: 3 },
                line: { color: color, width: 3 }
            };
        }

        function showCurve() {
            const dataset = datasets[Number(folderSelect.value)];
            const index = Number(curveSelect.value);
            const curve = dataset.curves[index];

            const traces = [
                {
                    x: curve.measured.map(point => point[0]),
                    y: curve.measured.map(point => point[1]),
                    type: "scatter",
                    mode: "lines+markers",
                    name: "Measured",
                    marker: {
                        color: "#58758a",
                        size: 4
                    },
                    line: {
                        color: "#58758a",
                        width: 1.5
                    }
                }
            ];

            if (curve.linear.length) {
                traces.push(
                    makeTrace(
                        curve.linear,
                        "Linear fit",
                        "#2ca02c",
                        "lines"
                    )
                );
            }

            if (curve.elasticity.length) {
                traces.push(
                    makeTrace(
                        curve.elasticity,
                        "Elasticity fit",
                        "#d62728",
                        "lines"
                    )
                );
            }

            const fittedContactX = Number(curve.values["contact_point_x_nm"]);
            const fittedContactY = Number(curve.values["contact_point_y_fit_nN"]);

            if (Number.isFinite(fittedContactX) && Number.isFinite(fittedContactY)) {
                traces.push({
                    x: [fittedContactX],
                    y: [fittedContactY],
                    type: "scatter",
                    mode: "markers",
                    name: "Fitted contact point",
                    marker: {
                        color: "#d62728",
                        size: 10,
                        symbol: "circle"
                    },
                    hovertemplate:
                        "Fitted contact point" +
                        "<br>X = %{x:.6g} nm" +
                        "<br>Force = %{y:.6g} nN" +
                        "<extra></extra>"
                });
            }

            Plotly.react(
                "plot",
                traces,
                {
                    title: `${dataset.name} | Curve ${curve.number}`,
                    xaxis: { title: "X (nm)", autorange: true },
                    yaxis: { title: "Force (nN)", autorange: true },
                    legend: { orientation: "h", y: -0.2 },
                    margin: { l: 70, r: 25, t: 60, b: 90 },
                    hovermode: "closest"
                },
                {
                    responsive: true,
                    displaylogo: false,
                    toImageButtonOptions: {
                        filename: `AFM_curve_${curve.number}`,
                        scale: 2
                    }
                }
            );

            valuesTable.replaceChildren();

            for (const [label, column, unit] of fields) {
                appendValueRow(
                    label,
                    formatNumber(curve.values[column]),
                    unit
                );

                if (column === "elasticity_rmsd_custom_pN") {
                    const rmsDifference = percentageDifference(
                        curve.values["elasticity_rms_bruker_pN"],
                        curve.values["elasticity_rmsd_custom_pN"]
                    );

                    appendValueRow(
                        "Elasticity RMS percentage difference",
                        rmsDifference,
                        "%"
                    );
                }
            }

            status.textContent =
                `Curve ${index + 1} of ${dataset.curves.length}. ` +
                `Position index: ${curve.position}.`;

            if (Object.keys(curve.values).length === 0) {
                status.textContent += " No matching summary row.";
            }

            filenameDisplay.textContent = curve.filename;

            document.getElementById("previous").disabled = index === 0;
            document.getElementById("next").disabled =
                index === dataset.curves.length - 1;
        }

        function showFolder() {
            const dataset = datasets[Number(folderSelect.value)];
            curveSelect.replaceChildren();

            dataset.curves.forEach((curve, index) => {
                curveSelect.add(
                    new Option(
                        `Curve ${curve.number} | Position ${curve.position}`,
                        index
                    )
                );
            });

            showCurve();
        }

        datasets.forEach((dataset, index) => {
            folderSelect.add(new Option(dataset.name, index));
        });

        folderSelect.onchange = showFolder;
        curveSelect.onchange = showCurve;

        document.getElementById("previous").onclick = () => {
            if (curveSelect.selectedIndex > 0) {
                curveSelect.selectedIndex -= 1;
                showCurve();
            }
        };

        document.getElementById("next").onclick = () => {
            if (curveSelect.selectedIndex < curveSelect.length - 1) {
                curveSelect.selectedIndex += 1;
                showCurve();
            }
        };

        if (window.Plotly) {
            showFolder();
        } else {
            status.textContent =
                "Could not load the plotting library. Check your connection and reload.";
        }
    </script>
</body>
</html>
"""


# ============================================================
# EMBED DATA AND WRITE index.html
# ============================================================

data_json = json.dumps(
    datasets,
    ensure_ascii=True,
    allow_nan=False,
).replace("<", "\\u003c")

final_html = (
    html
    .replace("__DATA__", data_json)
    .replace("__SUMMARY_TABLE__", summary_table_html)
)

output_file.write_text(
    final_html,
    encoding="utf-8",
)


# ============================================================
# CONSOLE SUMMARY
# ============================================================

print("Created:", output_file.resolve())
print("Folders:", len(datasets))
print("Curve CSVs:", sum(len(item["curves"]) for item in datasets))
print("Curve CSVs with unique summary matches:", len(matched_curve_summaries))
print("Raw summary rows (not automatically included):", total_summary_rows)
print(
    "RMS summary filter:",
    f"{RMS_COLUMN} <= {RMS_LIMIT_PN} pN"
)
print("Summary groups:")

for key in sorted(cell_stats):
    print(
        key,
        "n =",
        cell_stats[key]["n"],
    )


Summary match count: 20_cell1_crop1 | 19 matched per-curve CSVs | 0 summary entries not used
Summary match count: 20_cell1_crop2 | 12 matched per-curve CSVs | 0 summary entries not used
Summary match count: 20_cell1_crop3 | 19 matched per-curve CSVs | 1 summary entries not used
Summary match count: 20_cell2_crop1 | 30 matched per-curve CSVs | 0 summary entries not used
Summary match count: 20_cell2_crop2 | 23 matched per-curve CSVs | 0 summary entries not used
Summary match count: 20_cell3_crop1 | 28 matched per-curve CSVs | 0 summary entries not used
Summary match count: 20_cell3_crop2 | 22 matched per-curve CSVs | 0 summary entries not used
Summary match count: 320_cell1_crop1 | 4 matched per-curve CSVs | 0 summary entries not used
Summary match count: 320_cell1_crop2 | 8 matched per-curve CSVs | 1 summary entries not used
Summary match count: 320_cell1_crop3 | 27 matched per-curve CSVs | 4 summary entries not used
Summary match count: 320_cell1_crop4 | 11 matched per-curve CSVs | 4 

In [10]:
# AFM_ACCEPTED_CURVES_EXPORT_AUTOMATIC_V1
from pathlib import Path
import export_accepted_afm_curves as afm_export
afm_export.export_all(Path.cwd())


Exported 615 accepted curves.
  Live     : 153 curves across 11 crops
  20 mg/L  : 153 curves across 7 crops
  320 mg/L : 154 curves across 10 crops
  Dead     : 155 curves across 7 crops
  Rejected by five rules: 0
  Unmatched curve exports: 0
  Duplicate curve exports: 0
CSV files: /workspaces/AFM-plots_Jython_exports/AFM_accepted_curves_CSV
CSV zip: /workspaces/AFM-plots_Jython_exports/AFM_accepted_curves_CSV.zip
Excel workbook not created; artifact_tool is not available. Four CSVs were created.
